In [30]:
import json
from pathlib import Path

import numpy as np
from archai.discrete_search.algos import EvolutionParetoSearch
from archai.discrete_search.api import (
    SearchObjectives,
)
from kd_vs_hpo.nas.evaluators import PrecomputedArrayEvaluator
from nats_bench import create
from tqdm import tqdm

from kd_vs_hpo.nas.plots import plot_search_comparison

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
api = create("data/NATS-tss-v1_0-3ffb9.pickle.pbz2", "tss", fast_mode=False, verbose=False)
print("Total architectures:", len(api))

Total architectures: 15625


In [52]:
n = len(api.meta_archs)
dataset = "cifar100"
hp = "200"
n_eps = int(hp)
n_imgs = 50000
acc = np.zeros(n, dtype=np.float32)
flops = np.zeros(n, dtype=np.float32)
compute_time = np.zeros(n, dtype=np.float32)

for i in tqdm(range(n)):
    info = api.get_more_info(i, dataset, hp=hp, is_random=False)
    acc[i] = info["test-accuracy"]
    flops[i] = 3 * n_eps * n_imgs * api.arch2infos_dict[i][hp].get_compute_costs(dataset)["flops"] * 1e6
    compute_time[i] = info["train-all-time"]

best_model_ix = acc.argmax().item()
BEST_MODEL_FLOPS = int(flops[best_model_ix]
)



  0%|                                                                                                                                                               | 0/15625 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15625/15625 [00:21<00:00, 722.99it/s]


In [53]:
f"Best model FLOPs: {BEST_MODEL_FLOPS / 1e15} x 1e15 for HP {hp} with accuracy {acc[best_model_ix]:.4f} and compute time {compute_time[best_model_ix]:.2f} seconds. Dataset: {dataset}"

'Best model FLOPs: 5.542082991095808 x 1e15 for HP 200 with accuracy 73.5133 and compute time 5869.01 seconds. Dataset: cifar100'

In [54]:
objectives = SearchObjectives(cache_objective_evaluation=True)

objectives.add_objective(
    name="accuracy",
    model_evaluator=PrecomputedArrayEvaluator(acc, "accuracy"),
    higher_is_better=True,
    compute_intensive=False,
)

objectives.add_objective(
    name="flops",
    model_evaluator=PrecomputedArrayEvaluator(flops, "flops"),
    higher_is_better=False,
    compute_intensive=False,
)

In [32]:
INITIAL_INDICES = [1342, 8712, 11570] 

initial_dir = Path("./experiments/nas_initial_population")
initial_dir.mkdir(exist_ok=True)

initial_population_paths = []
for idx in INITIAL_INDICES:
    path = (initial_dir / str(idx)).with_suffix('.json')
    path.write_text(json.dumps({"nats_index": idx}))
    initial_population_paths.append(str(path))


In [55]:
from functools import partial

from kd_vs_hpo.nas.spaces import NATSEvolutionarySearchSpace

TOTAL_BUDGET = 40  # approximate number of model evaluations
N_SEEDS = 10


# ─── Evolution Pareto Search ─────────────────────────────────────
def run_evolution_search(seed: int, init_paths=None, tag="random"):
    # params = calculate_evolution_params(TOTAL_BUDGET)
    space = NATSEvolutionarySearchSpace(api, seed=seed)
    algo = EvolutionParetoSearch(
        search_space=space,
        search_objectives=objectives,
        output_dir=f"./out_evo/{tag}/{seed}",
        num_iters=4,
        initial_population_paths=init_paths,
        init_num_models=3,  # keep both variants at 3 initial models
        # num_random_mix=params["num_random_mix"],
        # mutations_per_parent=params["mutations_per_parent"],
        # num_crossovers=params["num_crossovers"],
        max_unseen_population=20,  # cap models evaluated per iteration
        seed=seed,
        save_pareto_model_weights=False,
    )
    results = algo.search()
    return results.get_search_state_df()

evolution_random = partial(run_evolution_search, init_paths=None, tag="random")
evolution_inited = partial(run_evolution_search, init_paths=initial_population_paths, tag="inited")


In [56]:
algorithms = [
    ("Evolution, random start", evolution_random, "rgba(214,39,40,1)"),
    ("Evolution, init from base archs", evolution_inited, "rgba(0,114,178,1)"),
]

fig = plot_search_comparison(
    algorithms=algorithms,
    acc=acc,
    flops=flops / 1e15, # just to make it readable
    compute_time=compute_time,
    n_seeds=N_SEEDS,
)


2026-11-11 22:15:39,963 - archai.discrete_search.algos.evolution_pareto — INFO —  Using 3 random architectures as the initial population ...
2026-11-11 22:15:39,964 - archai.discrete_search.algos.evolution_pareto — INFO —  Iteration 1/4
2026-11-11 22:15:39,964 - archai.discrete_search.algos.evolution_pareto — INFO —  Calculating search objectives ['accuracy', 'flops'] for 3 models ...
2026-11-11 22:15:39,965 - archai.discrete_search.algos.evolution_pareto — INFO —  Updating Pareto frontier ...
2026-11-11 22:15:39,965 - archai.discrete_search.algos.evolution_pareto — INFO —  Found 3 members.
2026-11-11 22:15:40,099 - archai.discrete_search.algos.evolution_pareto — INFO —  Optimzing memory usage ...
2026-11-11 22:15:40,100 - archai.discrete_search.algos.evolution_pareto — INFO —  Choosing 3 parents ...


Mutating parents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 11870.67it/s]

2026-11-11 22:15:40,102 - archai.discrete_search.algos.evolution_pareto — INFO —  Mutation: 3 new models.
2026-11-11 22:15:40,103 - archai.discrete_search.algos.evolution_pareto — INFO —  Crossover: 4 new models.
2026-11-11 22:15:40,104 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population: 12.
2026-11-11 22:15:40,105 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population after `max_unseen_population` restriction: 12.
2026-11-11 22:15:40,105 - archai.discrete_search.algos.evolution_pareto — INFO —  Iteration 2/4
2026-11-11 22:15:40,106 - archai.discrete_search.algos.evolution_pareto — INFO —  Calculating search objectives ['accuracy', 'flops'] for 12 models ...
2026-11-11 22:15:40,106 - archai.discrete_search.algos.evolution_pareto — INFO —  Updating Pareto frontier ...
2026-11-11 22:15:40,108 - archai.discrete_search.algos.evolution_pareto — INFO —  Found 4 members.
2026-11-11 22:15:40,285 - archai.discrete_search.algos.evolution_


Mutating parents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 10453.09it/s]

2026-11-11 22:15:40,380 - archai.discrete_search.algos.evolution_pareto — INFO —  Mutation: 4 new models.
2026-11-11 22:15:40,381 - archai.discrete_search.algos.evolution_pareto — INFO —  Crossover: 5 new models.
2026-11-11 22:15:40,382 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population: 14.
2026-11-11 22:15:40,382 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population after `max_unseen_population` restriction: 14.
2026-11-11 22:15:40,383 - archai.discrete_search.algos.evolution_pareto — INFO —  Iteration 3/4
2026-11-11 22:15:40,384 - archai.discrete_search.algos.evolution_pareto — INFO —  Calculating search objectives ['accuracy', 'flops'] for 14 models ...
2026-11-11 22:15:40,393 - archai.discrete_search.algos.evolution_pareto — INFO —  Updating Pareto frontier ...
2026-11-11 22:15:40,476 - archai.discrete_search.algos.evolution_pareto — INFO —  Found 5 members.


2026-11-11 22:15:40,625 - archai.discrete_search.algos.evolution_pareto — INFO —  Optimzing memory usage ...
2026-11-11 22:15:40,626 - archai.discrete_search.algos.evolution_pareto — INFO —  Choosing 5 parents ...


... (223 outputs left out)

2026-11-11 22:15:53,743 - archai.discrete_search.algos.evolution_pareto — INFO —  Mutation: 5 new models.
2026-11-11 22:15:53,744 - archai.discrete_search.algos.evolution_pareto — INFO —  Crossover: 3 new models.
2026-11-11 22:15:53,745 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population: 13.
2026-11-11 22:15:53,745 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population after `max_unseen_population` restriction: 13.
2026-11-11 22:15:53,745 - archai.discrete_search.algos.evolution_pareto — INFO —  Iteration 4/4
2026-11-11 22:15:53,746 - archai.discrete_search.algos.evolution_pareto — INFO —  Calculating search objectives ['accuracy', 'flops'] for 13 models ...
2026-11-11 22:15:53,747 - archai.discrete_search.algos.evolution_pareto — INFO —  Updating Pareto frontier ...
2026-11-11 22:15:53,751 - archai.discrete_search.algos.evolution_pareto — INFO —  Found 6 members.


2026-11-11 22:15:53,913 - archai.discrete_search.algos.evolution_pareto — INFO —  Optimzing memory usage ...
2026-11-11 22:15:53,914 - archai.discrete_search.algos.evolution_pareto — INFO —  Choosing 6 parents ...


Mutating parents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 17537.16it/s]

2026-11-11 22:15:53,916 - archai.discrete_search.algos.evolution_pareto — INFO —  Mutation: 6 new models.
2026-11-11 22:15:53,917 - archai.discrete_search.algos.evolution_pareto — INFO —  Crossover: 4 new models.
2026-11-11 22:15:53,918 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population: 15.
2026-11-11 22:15:53,919 - archai.discrete_search.algos.evolution_pareto — INFO —  Total unseen population after `max_unseen_population` restriction: 15.


In [60]:
import plotly.graph_objects as go

fig_copy = go.Figure(fig)

In [61]:
fig_copy.add_vline(x=4298215680000000/1e15, col=2, line=dict(width=2, dash='dash'))
fig_copy.add_trace(
            go.Scatter(
                x=[None], y=[None],
                mode="lines",
                name="Best base accuracy",
                line=dict(color="lightgreen", dash="dash", width=2),
            ),
            row=1,
            col=2
        )

# fig_copy.add_hline(
#             y=acc[11570], line_dash="dash", line_color="lightgreen", row=1, col=2
#         )

fig_copy.update_layout(
    title={
        'text': "Evolution architecture search on CIFAR-100",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    }
)